<a href="https://colab.research.google.com/github/lucmos2002/workshopGITHUB/blob/master/02x_data_Lucas_Souza.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Preparação de Dados

## 1 Byte pair encoding de palavras fora do voculário

Durante a aula vimos que um tokenizador baseado em Byte pair encoding (BPE) é capaz de lidar com palavras fora do vocabulário ao dividir uma palavra em "sub-palavras" que estejam presentes no vocabulário. Na pior das hipóteses a palavra pode ser quebrada em letras individuais.

O texto abaixo é um trecho tirado do primeiro parágrafo do livro "The Time Machine" (H. G. Wells, 1895). Use o Tiktoken (com encoding do gpt2) visto durante a aula para tokenizá-lo e verifique quais palavras não estão presentes no vocabulário e necessitaram ser quebradas em "sub-palavras". Mostre como ficou a divisão de cada uma das palavras originalmente fora do vocabulário após a tokenização.

Por exemplo, a palavra "luxurious":<br>
`luxurious -> ['lux', 'urious']`

In [ ]:
time_machine_text = 'The Time Traveller was expounding a recondite matter to us. \
His grey eyes shone and twinkled, and his usually pale face was flushed and animated.'

In [ ]:
import tiktoken

# SEU CÓDIGO AQUI
encoding = tiktoken.encoding_for_model("gpt-2")
token_ids= encoding.encode(time_machine_text)
print(token_ids)

[464, 3862, 43662, 6051, 373, 1033, 9969, 257, 664, 623, 578, 2300, 284, 514, 13, 2399, 13791, 2951, 44193, 290, 665, 676, 992, 11, 290, 465, 3221, 14005, 1986, 373, 44869, 290, 15108, 13]


## 2 Data loader com diferentes tamanhos de contexto e strides

Durante a aula, vimos como criar um data loader pra treinar uma LLM através da tarefa de prever o próximo token. No caso, o input `x` é uma sequência de tokens e o alvo `y` é o próximo token da sequência `x`. O data loader cria uma janela deslizante que percorre todo o texto, gerando inúmeros exemplos de treino `x, y`. A quantidade de dados de treino gerada pelo data loader vai variar de acordo com o tamanho de `x` (`max_length`) e o tanto que a janela irá deslizar (`stride`) ao longo do texto.

Use o data loader visto durante a aula para tokenizar o texto abaixo com duas configurações distintas:
- `batch_size=4, max_length=2, stride=1`
- `batch_size=4, max_length=6, stride=2`

E responda, quantos exemplos de treino cada configuração o data loader gerou? Lembre-se que cada batch pode conter até 4 exemplos de treino.

In [ ]:
time_machine_text = "The Time Traveller (for so it will be convenient to speak of him) \
was expounding a recondite matter to us. His grey eyes shone and \
twinkled, and his usually pale face was flushed and animated. The \
fire burned brightly, and the soft radiance of the incandescent \
lights in the lilies of silver caught the bubbles that flashed and \
passed in our glasses. Our chairs, being his patents, embraced and \
caressed us rather than submitted to be sat upon, and there was that \
luxurious after-dinner atmosphere when thought roams gracefully \
free of the trammels of precision. And he put it to us in this \
way--marking the points with a lean forefinger--as we sat and lazily \
admired his earnestness over this new paradox (as we thought it) \
and his fecundity."

In [ ]:
import tiktoken
import torch
from torch.utils.data import Dataset, DataLoader

# SEU CÓDIGO COM A CLASSE DO DATASET E A FUNÇÃO DO DATA LOADER
class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize the entire text as a single sequence of IDs
        token_ids = tokenizer.encode(txt)

        # Use a sliding window to chunk the book into overlapping sequences of max_length
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [ ]:
def create_dataloader_v1(txt, batch_size=4, max_length=256, stride=128, shuffle=True, drop_last=True, num_workers=0):

    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Create dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

In [23]:
# CHAME O DATA LOADER COM TEXTO ACIMA COM A 1ª CONFIGURAÇÃO
# E CONTE OS EXEMPLOS DE TREINO
dataloader = create_dataloader_v1(time_machine_text, batch_size=4, max_length=2, stride=1, shuffle=False)

total_examples = 0
for inputs, targets in dataloader:
  total_examples += len(inputs)
print("Exemplos de treino:", total_examples)


Exemplos de treino: 168


In [24]:
for inputs, targets in dataloader:
  print("Inputs:\n", inputs)
  print("\nTargets:\n", targets)

Inputs:
 tensor([[  464,  3862],
        [ 3862, 43662],
        [43662,  6051],
        [ 6051,   357]])

Targets:
 tensor([[ 3862, 43662],
        [43662,  6051],
        [ 6051,   357],
        [  357,  1640]])
Inputs:
 tensor([[ 357, 1640],
        [1640,  523],
        [ 523,  340],
        [ 340,  481]])

Targets:
 tensor([[1640,  523],
        [ 523,  340],
        [ 340,  481],
        [ 481,  307]])
Inputs:
 tensor([[  481,   307],
        [  307, 11282],
        [11282,   284],
        [  284,  2740]])

Targets:
 tensor([[  307, 11282],
        [11282,   284],
        [  284,  2740],
        [ 2740,   286]])
Inputs:
 tensor([[2740,  286],
        [ 286,  683],
        [ 683,    8],
        [   8,  373]])

Targets:
 tensor([[ 286,  683],
        [ 683,    8],
        [   8,  373],
        [ 373, 1033]])
Inputs:
 tensor([[ 373, 1033],
        [1033, 9969],
        [9969,  257],
        [ 257,  664]])

Targets:
 tensor([[1033, 9969],
        [9969,  257],
        [ 257,  664],
 

In [25]:
# CHAME O DATA LOADER COM TEXTO ACIMA COM A 2ª CONFIGURAÇÃO
# E CONTE OS EXEMPLOS DE TREINO
dataloader = create_dataloader_v1(time_machine_text, batch_size=4, max_length=6, stride=2, shuffle=False)

total_examples = 0
for inputs, targets in dataloader:
  total_examples += len(inputs)
print("Exemplos de treino:", total_examples)

Exemplos de treino: 80


In [26]:
for inputs, targets in dataloader:
  print("Inputs:\n", inputs)
  print("\nTargets:\n", targets)

Inputs:
 tensor([[  464,  3862, 43662,  6051,   357,  1640],
        [43662,  6051,   357,  1640,   523,   340],
        [  357,  1640,   523,   340,   481,   307],
        [  523,   340,   481,   307, 11282,   284]])

Targets:
 tensor([[ 3862, 43662,  6051,   357,  1640,   523],
        [ 6051,   357,  1640,   523,   340,   481],
        [ 1640,   523,   340,   481,   307, 11282],
        [  340,   481,   307, 11282,   284,  2740]])
Inputs:
 tensor([[  481,   307, 11282,   284,  2740,   286],
        [11282,   284,  2740,   286,   683,     8],
        [ 2740,   286,   683,     8,   373,  1033],
        [  683,     8,   373,  1033,  9969,   257]])

Targets:
 tensor([[  307, 11282,   284,  2740,   286,   683],
        [  284,  2740,   286,   683,     8,   373],
        [  286,   683,     8,   373,  1033,  9969],
        [    8,   373,  1033,  9969,   257,   664]])
Inputs:
 tensor([[ 373, 1033, 9969,  257,  664,  623],
        [9969,  257,  664,  623,  578, 2300],
        [ 664,  623,  5